In [ ]:
from models import get_models
from preprocess import load_splits
import matplotlib.pyplot as plt
import pandas as pd
from src.models import get_models, cross_validate_models, evaluate, soft_rank_models, tune_models, export_predictions

X_train, X_val, X_test, y_train, y_val, y_test = load_splits()
models=get_models()

df = X_train.copy()
df["default"] = y_train

print(y_train.value_counts(normalize=True))

X_train.hist(figsize=(16, 12), bins=30)
plt.tight_layout()
plt.show()

In [ ]:
corr = df.corr()["default"].drop("default").sort_values()
corr.plot(kind="barh", figsize=(7, 8), title="Correlation with default")
plt.show()

In [ ]:
from sklearn.inspection import permutation_importance
from src.config import SEED

model = models["random_forest"].fit(X_train, y_train)
r = permutation_importance(model, X_val, y_val, scoring="average_precision",
                           n_repeats=10, random_state=SEED, n_jobs=1)
imp = pd.Series(r.importances_mean, index=X_val.columns).sort_values()
imp.plot(kind="barh", figsize=(7, 8), title="Permutation importance (validation PR-AUC drop)")
plt.show()

In [ ]:
results = {}
for name, m in models.items():
    m.fit(X_train, y_train)
    results[name] = evaluate(m, X_val, y_val)

res = pd.DataFrame(results).T[["auc", "pr_auc"]]
ax = res.plot(kind="bar", figsize=(8, 5), rot=0)
ax.set_ylabel("score (higher is better)")
ax.set_title("Validation performance")
ax.legend(["AUC", "PR-AUC"])
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(9, 8))
for ax, (name, m) in zip(axes.ravel(), models.items()):
    ConfusionMatrixDisplay.from_estimator(
        m, X_val, y_val, ax=ax, colorbar=False,
        display_labels=["no default", "default"], cmap="Blues")
    ax.set_title(name)
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import precision_score, recall_score
rows = {}
for name, m in models.items():
    p = m.predict(X_val)
    rows[name] = {"precision": precision_score(y_val, p), "recall": recall_score(y_val, p)}
print((pd.DataFrame(rows).T * 100).round(0).astype(int).astype(str) + "%")

In [ ]:
before = pd.Series({"logreg": 0.461, "decision_tree": 0.456,
                    "random_forest": 0.477, "hist_bg": 0.455})
cv = cross_validate_models(models, X_train, y_train)
after = cv["cv_pr_auc"]
order = ["logreg", "decision_tree", "random_forest", "hist_bg"]

cmp = pd.DataFrame({"default settings": before, "tuned": after}).loc[order]
ax = cmp.plot(kind="bar", figsize=(8, 5), rot=0)
for c in ax.containers:
    ax.bar_label(c, fmt="%.2f", fontsize=9)
ax.set_ylim(0.4, 0.55)
ax.set_ylabel("cross-validated score")
ax.set_title("Effect of tuning (5-fold CV, training data)")
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier, plot_tree

plt.style.use("default")   # light background, matches your other slides

demo = DecisionTreeClassifier(max_depth=3, min_samples_leaf=91,
                              class_weight="balanced", random_state=42)
demo.fit(X_train, y_train)   # raw columns; trees don't need scaling

fig, ax = plt.subplots(figsize=(14, 6))
plot_tree(demo, feature_names=list(X_train.columns),
          class_names=["no default", "default"],
          filled=True, rounded=True, impurity=False,
          fontsize=9, ax=ax)
plt.tight_layout()
fig.savefig("decision_tree.png", dpi=200, bbox_inches="tight")
plt.show()